# PULSAR Benchmark Dataset Plotter & Inspector
This notebook loads, inspects, and visualizes synthetic radar pulse datasets generated by `generate_datasets.py`.

It works for both **single-radar** and **interleaved multi-radar** datasets across all error levels.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

%matplotlib inline
plt.style.use("seaborn-v0_8-whitegrid")

### 1. Locate Datasets
Scans `../datasets/` for all available generated datasets.

In [ ]:
PROJECT_ROOT = Path("..").resolve()
DATASETS_ROOT = PROJECT_ROOT / "datasets"

def discover_datasets(root: Path) -> list[Path]:
    if not root.exists():
        return []
    # Any folder containing metadata.yaml is a valid dataset
    return sorted([p.parent for p in root.rglob("metadata.yaml")])

available = discover_datasets(DATASETS_ROOT)
print(f"Found {len(available)} datasets under {DATASETS_ROOT}:")
for i, p in enumerate(available):
    print(f"  [{i}] {p.relative_to(DATASETS_ROOT)}")

# Select a dataset (defaults to the first discovered, or fallback to an example)
if available:
    SELECTED_DATASET = available[0]
else:
    SELECTED_DATASET = DATASETS_ROOT / "single_radar" / "fixed" / "single_stable" / "single_stable_10x128_clean"

print(f"\nSelected Dataset: {SELECTED_DATASET}")

### 2. View Dataset Metadata

In [ ]:
meta_path = SELECTED_DATASET / "metadata.yaml"
if meta_path.exists():
    with open(meta_path, "r", encoding="utf-8") as f:
        meta = yaml.safe_load(f)
    for k, v in meta.items():
        print(f"{k}: {v}")
else:
    print(f"No metadata.yaml found at {meta_path}")

### 3. Plot a Single Sequence (4-Panel PDW Diagnostics)

In [ ]:
def load_sequence(dataset_dir: Path, split: str = "train", seq_id: str = "seq_0000") -> pd.DataFrame:
    csv_path = dataset_dir / split / f"{seq_id}.csv"
    if not csv_path.exists():
        # Fallback to combined sequences.csv
        comb_path = dataset_dir / split / "sequences.csv"
        if comb_path.exists():
            df = pd.read_csv(comb_path)
            seq_df = df[df["sequence_id"] == seq_id].copy()
            if not seq_df.empty:
                return seq_df.sort_values("pulse_index").reset_index(drop=True)
        raise FileNotFoundError(f"Sequence {seq_id} not found in {dataset_dir / split}")
    return pd.read_csv(csv_path).sort_values("pulse_index").reset_index(drop=True)

def plot_sequence(seq_df: pd.DataFrame):
    seq_id = seq_df["sequence_id"].iloc[0] if "sequence_id" in seq_df.columns else "Unknown"
    radar_type = seq_df["radar_type"].iloc[0] if "radar_type" in seq_df.columns else "Radar"
    err_level = seq_df["error_level"].iloc[0] if "error_level" in seq_df.columns else "Unknown"

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle(f"Sequence: {seq_id} | Type: {radar_type} ({err_level}) | {len(seq_df)} pulses", fontsize=15, fontweight='bold')

    # 1. TOA
    axes[0, 0].plot(seq_df["pulse_index"], seq_df["TOA"], "o-", color="tab:blue", markersize=3, linewidth=1.2)
    axes[0, 0].set_title("TOA vs Pulse Index")
    axes[0, 0].set_xlabel("Pulse Index (n)")
    axes[0, 0].set_ylabel("TOA (\u03bcs)")

    # 2. Measured Delta-TOA (PRI)
    axes[0, 1].plot(seq_df["pulse_index"], seq_df["Measured_Delta_TOA"], "s-", color="tab:orange", markersize=3, linewidth=1.2, label="Measured \u0394TOA")
    if "PRI" in seq_df.columns and not seq_df["PRI"].isna().all():
        axes[0, 1].plot(seq_df["pulse_index"], seq_df["PRI"], "--", color="tab:red", alpha=0.6, label="Source PRI")
    axes[0, 1].set_title("Measured Delta-TOA (PRI Estimation)")
    axes[0, 1].set_xlabel("Pulse Index (n)")
    axes[0, 1].set_ylabel("\u0394TOA (\u03bcs)")
    axes[0, 1].legend()

    # 3. Frequency
    axes[1, 0].plot(seq_df["pulse_index"], seq_df["Frequency"], "^-", color="tab:green", markersize=3, linewidth=1.2)
    axes[1, 0].set_title("Carrier Frequency vs Pulse Index")
    axes[1, 0].set_xlabel("Pulse Index (n)")
    axes[1, 0].set_ylabel("Frequency (MHz)")

    # 4. Polar AOA / Amplitude
    ax_polar = fig.add_subplot(2, 2, 4, projection="polar")
    theta = np.deg2rad(seq_df["AOA"])
    sc = ax_polar.scatter(theta, seq_df["Amplitude"], c=seq_df["Amplitude"], cmap="viridis", s=35, edgecolors="black", linewidths=0.3)
    ax_polar.set_title("AOA vs Amplitude")
    ax_polar.set_theta_zero_location("E")
    ax_polar.set_theta_direction(1)
    ax_polar.set_rlabel_position(90)
    fig.colorbar(sc, ax=ax_polar, pad=0.08, label="Amplitude (dBm)")

    plt.tight_layout()
    plt.show()

if SELECTED_DATASET.exists():
    seq = load_sequence(SELECTED_DATASET, split="train", seq_id="seq_0000")
    plot_sequence(seq)

### 4. Multi-Sequence Grid Plot
Inspect the first 6 sequences in the train split side-by-side.

In [ ]:
def plot_sequence_grid(dataset_dir: Path, split: str = "train", max_seqs: int = 6):
    split_dir = dataset_dir / split
    seq_files = sorted(list(split_dir.glob("seq_*.csv")))[:max_seqs]
    if not seq_files:
        print(f"No seq_*.csv files in {split_dir}")
        return

    cols = 3
    rows = (len(seq_files) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 3.5 * rows))
    axes = np.array(axes).reshape(-1)

    for i, f in enumerate(seq_files):
        df = pd.read_csv(f)
        axes[i].plot(df["pulse_index"], df["Measured_Delta_TOA"], "o-", markersize=2, linewidth=1)
        axes[i].set_title(f.stem)
        axes[i].set_xlabel("Pulse Index")
        axes[i].set_ylabel("\u0394TOA (\u03bcs)")

    for j in range(len(seq_files), len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()

if SELECTED_DATASET.exists():
    plot_sequence_grid(SELECTED_DATASET, split="train", max_seqs=6)